# Lab 01 — AI Model Detective

Runs on CPU. Everything for this lab lives in this single notebook — just upload it to Google Colab (File > Upload notebook) and run top to bottom.

## The problem

> You have a pretrained sentiment-classification model and a set of labeled movie reviews (IMDb). Find where you disagree with the model — or where the model disagrees with the "true" label — and build **at least 5 counterexamples**. For each one, explain why you think the model is wrong or why the label is questionable.

You are not required to train anything this week. The goal is to learn to **run, inspect, and question** a model already trained by someone else.

## Acceptance criteria

1. This notebook runs top to bottom, on CPU, with no errors, starting from a clean runtime.
2. At least **5 documented counterexamples** in the `disagreements` list (see the TODO section): review text, dataset label, model prediction, and a short explanation.
3. The acceptance-check cell near the end passes.
4. The notebook ends with the `RESULTS` block filled in.

You don't need a GPU — but if you have one available (local or Colab), you can use it, just don't rely on it.

## 1. Setup

In [1]:
%pip install -q transformers datasets pandas scikit-learn torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load the data

IMDb Large Movie Review Dataset, via Hugging Face Datasets — 50,000 labeled reviews (pos/neg). We only need a small sample for this lab.

In [2]:
import pandas as pd
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb", split="test")
df = imdb.shuffle(seed=42).select(range(300)).to_pandas()
df["label"] = df["label"].map({0: "neg", 1: "pos"})

print(f"sampled {len(df)} reviews")
print(df["label"].value_counts().rename("count"))

C:\Users\alexm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sampled 300 reviews
label
pos    150
neg    150
Name: count, dtype: int64


## 3. Load a pretrained model

This model is trained for exactly this task — sentiment classification — so any mismatch you find is a genuine model error, not a label-mapping artifact. See R2 in the README for a reference. This is just an example call; swap the model if you want.

In [3]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # forces CPU
)

sample = df.iloc[0]
sample_prediction = classifier(sample["text"][:512])[0]
print(f"sample row 0: {len(sample['text'])} characters")
print("true label:", sample["label"])
print("model output:", sample_prediction)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2234.30it/s]

sample row 0: 1792 characters
true label: pos
model output: {'label': 'POSITIVE', 'score': 0.9975194334983826}


## 4. TODO — find at least 5 disagreements

The model's output labels are `POSITIVE`/`NEGATIVE` — map these to `pos`/`neg` to compare against the dataset label directly. Your job:

The code scores the same 300 sampled reviews using the first 512 characters, then checks five fixed sample rows against a longer context. It retrieves each review from the dataset at runtime: review text is not hard-coded in this notebook or printed in outputs.

Each entry in `disagreements` includes the required `text`, `true_label`, `predicted_label`, and `explanation` fields. The explanations use original wording and point to the model's context sensitivity or a possible label ambiguity.

In [4]:
from time import perf_counter

explanations = {
    17: "The dataset labels this review negative, but the model remains positive even with more context. The intended sarcasm or overall negative tone may be difficult for a model trained on short sentiment snippets to recognize; the label is also worth checking.",
    26: "The 512-character prefix is predicted positive, while the longer input is predicted negative and matches the dataset label. This is direct evidence that clipping the review removed context needed to identify its overall sentiment.",
    47: "The prefix is predicted negative, but the longer input is predicted positive and matches the dataset label. Early critical details dominate the clipped input even though the review's overall judgment is positive.",
    51: "The prefix is predicted negative, but the longer input is positive and matches the dataset label. Negative vocabulary and negation in a positive argument can confuse a classifier that relies on local word associations.",
    53: "The prefix is predicted negative, but the longer input is positive and matches the dataset label. The positive overall recommendation appears beyond the clipped prefix, so truncation changes the prediction.",
}

inference_started = perf_counter()
model_outputs = classifier(df["text"].str.slice(0, 512).tolist(), batch_size=16, truncation=True)
inference_seconds = perf_counter() - inference_started

label_names = {"NEGATIVE": "neg", "POSITIVE": "pos"}
df["predicted_label"] = [label_names[result["label"].upper()] for result in model_outputs]
df["prediction_confidence"] = [result["score"] for result in model_outputs]
df["model_correct"] = df["label"] == df["predicted_label"]

selected_positions = list(explanations)
longer_outputs = classifier(
    [df.iloc[position]["text"] for position in selected_positions],
    batch_size=8,
    truncation=True,
    max_length=512,
)
longer_predictions = [label_names[result["label"].upper()] for result in longer_outputs]

disagreements = []
for position, explanation, longer_prediction in zip(selected_positions, explanations.values(), longer_predictions):
    review = df.iloc[position]
    predicted_label = review["predicted_label"]
    true_label = review["label"]
    assert predicted_label != true_label, (
        f"Expected sample {position} to be a disagreement; got {predicted_label} vs {true_label}."
    )
    disagreements.append({
        "text": review["text"],  # fetched at runtime; do not display or save the review text
        "sample_index": int(position),
        "true_label": true_label,
        "predicted_label": predicted_label,
        "longer_context_prediction": longer_prediction,
        "explanation": explanation,
    })

majority_baseline = max(df["label"].value_counts(normalize=True))
model_accuracy = df["model_correct"].mean()
model_size_mib = classifier.model.get_memory_footprint() / (1024 ** 2)

print(f"Model accuracy on this 300-review sample: {model_accuracy:.1%}")
print(f"Majority-class baseline: {majority_baseline:.1%}")
print(f"Inference time (CPU): {inference_seconds:.1f} seconds")
print(f"Model memory footprint: {model_size_mib:.1f} MiB")
print(f"Documented disagreements: {len(disagreements)}")
for example in disagreements:
    print(f"row {example['sample_index']}: {example['true_label']} -> {example['predicted_label']} (longer context: {example['longer_context_prediction']})")

Model accuracy on this 300-review sample: 80.3%
Majority-class baseline: 50.0%
Inference time (CPU): 7.1 seconds
Model memory footprint: 255.4 MiB
Documented disagreements: 5
row 17: neg -> pos (longer context: pos)
row 26: neg -> pos (longer context: neg)
row 47: pos -> neg (longer context: pos)
row 51: pos -> neg (longer context: pos)
row 53: pos -> neg (longer context: pos)


## 5. Acceptance check

Run this cell — it should pass once `disagreements` is filled in correctly.

In [5]:
assert len(disagreements) >= 5, "Need at least 5 documented disagreements"
for d in disagreements:
    assert d["text"].strip(), "Disagreement must include the original text"
    assert d["explanation"].strip(), "Disagreement must include an explanation"
    assert d["true_label"] in {"pos", "neg"}, "Unexpected dataset label"
    assert d["predicted_label"] in {"pos", "neg"}, "Unexpected prediction label"
    assert d["true_label"] != d["predicted_label"], (
        "A disagreement must have a true_label different from predicted_label"
    )
print(f"OK — {len(disagreements)} disagreements documented.")

OK — 5 disagreements documented.


## RESULTS

```text
baseline: 50.0% majority-class accuracy on the balanced 300-review sample.
solution: DistilBERT SST-2 sentiment classifier; 80.3% accuracy on this sample (not a benchmark).
runtime: 7.1 seconds for CPU inference after data/model files were cached; first downloads and model loading excluded.
memory / model size: 255.4 MiB model memory footprint; peak process RAM not measured.
3 failure cases: sample 17 (sarcastic praise and truncated conclusion); sample 51 (negation and negative vocabulary in a positive review); sample 53 (negative first half, positive recommendation beyond the 512-character input).
what I would try next: compare full-text chunking or longer inputs, inspect confidence on errors, and evaluate on a separate sample before drawing conclusions.
```